In [ ]:
%run board.ipynb
%run MCTS.ipynb
%run UI.ipynb

### PopOut Arena!

Como parte do processo de avaliação de algoritmos pensados pela equipa, foi desenvolvido uma arena para que seja possível realizar muitos testes entre os diversos modelos construídos.

Graças a separação da classe UI das demais peças técnicas do jogo, é possível simular o jogo sem qualquer tipo de input ou output no terminal, permitindo assim que diversas simulações possam ser feitas em sequência.

A primeira função principal é *simulate_match*. Ela recebe o nome e a função de decisão dos dois jogadores, cria um tabuleiro novo e executa uma partida completa entre eles.

As funções para MCTS são padronizadas para c = 1.414, e número de iterações sendo 2000, considerado pela equipa um bom número para testar a qualidade dos algoritmos, um número de iterações menor poderia resultar na aproximação dos produtos dos algoritmos não dando iterações suficientes para que eles se destaquem por suas qualidades.

A função retorna o nome do vencedor ou *Draw* caso a partida termine empatada.

In [ ]:
def simulate_match(p1_name, p1_func, p2_name, p2_func):
    board = Board()
    state_history = {board.get_state(): 1}
    
    while True:
        piece = board.current_player
        opponent_piece = 'O' if piece == 'X' else 'X'
        
        current_func = p1_func if piece == 'X' else p2_func
        current_name = p1_name if piece == 'X' else p2_name
        opponent_name = p2_name if piece == 'X' else p1_name

        current_state = board.get_state()
        if state_history.get(current_state, 0) >= 3:
            return "Draw"
            
        move = current_func(board)
        col = move[1]

        if move[0] == "pop":
            board.pop_piece(col)
            cw = board.check_win(piece)
            ow = board.check_win(opponent_piece)
            if cw and ow: return current_name 
            elif cw: return current_name
            elif ow: return opponent_name
        elif move[0] == "push":
            board.drop_piece(col, piece)
            if board.check_win(piece): return current_name

        new_state = board.get_state()
        state_history[new_state] = state_history.get(new_state, 0) + 1
        
        if not board.get_valid_moves():
            return "Draw"
            
        board.switch_player()

A função *run_combat_arena* organiza vários confrontos entre os modelos. Ela define três versões de jogadores: *MCTS Vanilla*, *MCTS Heurístico* e *MCTS Multi-Expansion*.

Depois disso, são criados alguns confrontos entre esses modelos. O confronto *Vanilla vs Heurística* aparece também invertido como *Heurística vs Vanilla*. Isso é intencional, pois considerou a possibilidade de que o primeiro a jogar tivesse algum tipo de vantagem sobre o segundo player, logo para fins de maior transparência e clareza na decisão, as simulações também serão executadas com a ordem dos jogadores trocadas. Ex: Para cada jogo X vs Y, haverá um jogo Y vs X.

Para cada confronto, o código executa várias partidas, conta quantas vitórias cada jogador teve e também quantos empates aconteceram.

No final, a função imprime uma tabela simples com os resultados finais de cada combate.

In [ ]:
import time
import itertools

def run_combat_arena(num_games, iterations, n_children):
    heuristica = lambda b: mcts_best_move(b, iterations=iterations)
    vanilla = lambda b: mcts_vanilla_best_move(b, iterations=iterations)
    multi = lambda b: mcts_multi_expansion_best_move(b, iterations=iterations, n_children=n_children)

    nome_multi = f"Multi({n_children})"

    modelos = [
        ("Vanilla", vanilla),
        ("Heurística", heuristica),
        (nome_multi, multi)
    ]

    matchups = [
        (p1[0], p1[1], p2[0], p2[1]) 
        for p1, p2 in itertools.permutations(modelos, 2)
    ]

    results_table = []

    for p1_name, p1_func, p2_name, p2_func in matchups:
        print(f"A simular: [X] {p1_name} vs [O] {p2_name}... ", end="", flush=True)
        
        start_time = time.time()
        p1_wins, p2_wins, draws = 0, 0, 0
        
        for i in range(num_games):
            winner = simulate_match(p1_name, p1_func, p2_name, p2_func)
            if winner == p1_name: p1_wins += 1
            elif winner == p2_name: p2_wins += 1
            else: draws += 1
                
        print(f"✅ ({time.time() - start_time:.1f}s)")
        
        results_table.append({
            "Match": f"{p1_name} vs {p2_name}", "P1": p1_wins, "P2": p2_wins, "D": draws
        })

    return results_table

Por fim, o número de jogos a serem jogados entre cada uma das opções de batalha e o número de iterações dos MCTS é definido na função *start_arena*, a qual recebe os resultados e os transforma numa tabela formatada utilizando a biblioteca pandas.

Ao chamar a função, pode-se observar os resultados.

In [ ]:
import pandas as pd
from IPython.display import display, HTML

def start_arena(num_games, iterations, n_children_list=[3, 5, 7]):
    print(f"⚔️ ARENA ATIVA: {num_games} jogos por confronto ({iterations} iterações) ⚔️\n")
    
    todos_os_dataframes = []
    
    for n in n_children_list:
        print(f"\n{'='*45}")
        print(f"🏆 INICIANDO TORNEIO: MULTI ({n} CHILDREN) 🏆")
        print(f"{'='*45}")
        
        resultados_brutos = run_combat_arena(num_games=num_games, iterations=iterations, n_children=n)
        
        df = pd.DataFrame(resultados_brutos)
        df = df.rename(columns={
            "Match": "Confronto", 
            "P1": "Vitórias P1", 
            "P2": "Vitórias P2", 
            "D": "Empates"
        })
        
        df.insert(0, "Cenário", f"Multi({n})")
        
        display(HTML(f"<br><b>Tabela Final - Torneio Multi({n}):</b>"))
        display(df)
        
        todos_os_dataframes.append(df)
        
    print("\n✅ TODAS AS SIMULAÇÕES FORAM CONCLUÍDAS!")

children_list = [1, 3, 5, 7, 9, 11]

#df_final = start_arena(num_games=20, iterations=10000, n_children_list=children_list)
#display(df_final)

Através da visualização dos resultados da arena, a equipa identificou que a multi-expansão de nós não é tão efetiva quanto a simples inserção de heurísticas no algoritmo MonteCarlo. Então, tendo selecionado o algiritmo campeão, inicou-se a etapa da criação do dataset.

### Criação de DataSets

O dataset necessário para treinar um modelo de Árvore de Decisão deve ser composto por N colunas contendo informações sobre cada dado mas apenas 1 única coluna label para fins de classificação.

Conforme documentação no Assingment, o formato do dataset é um tuplo (estado do board, movimentação) onde a útlima coluna(label) representará a movimentação escolhida pelo algoritmo.

Então, abaixo estão as estratégias utilizadas para criação do dataset:
- estado do board: o estado do board foi convertido em 42 variáveis (representando cada célula disponível do tabuleiro), entituladas *c_row_col*. Quanto ao conteúdo de cada variável, se está vazia, o valor guardado é *V*. Caso contrário, guarda-se a peça presente naquela posição, ou seja, *X* ou *O*. A função *get_flat_board* foi pensada para isso, transformar o estado do board em uma lista de 42 posições, cada um deles com seu devido preenchidmento representando um estado do jogo.

- movimentação: para cada estado(conjunto de 42 variáveis), o movimento escolhido ao final do algoritmo MCTS foi atirbuido a coluna label. Simples assim.

In [ ]:
def get_flat_board(board):
    """Esmaga o tabuleiro numa lista linear de 42 strings ('X', 'O', 'V')."""
    flat = []
    for r in range(board.rows):
        for c in range(board.cols):
            piece = board.grid[r][c]
            if piece == ' ' or piece == 0 or piece is None:
                flat.append('V')
            else:
                flat.append(piece)
    return flat

Antes de criar o dataset, considerando ainda a natureza probabilística do algoritmo MonteCarlo e com a intenção de maximizar ao máximo a qualidade dos dados de treino, a equipa optou apenas por guardar as jogadas no **jogador vencedor**. Então, foi definida a função *generate_dataset_winnsers_only*.

Durante cada partida, a função não escreve imediatamente no ficheiro de formato CSV. Primeiro ele guarda as jogadas em uma memória temporária chamada *game_memory*. E apenas posterior a um resultado alcançado (empates excluídos), grava os estados e as suas respectivas jogadas feitas pelo jogador vencedor.

In [ ]:
import csv

def generate_dataset_winners_only(model_func, model_name, num_games, iterations, filename):
    print(f"\n🧠 A gerar dataset de ALTA QUALIDADE usando {model_name} 🧠")
    print(f"-> Apenas as jogadas do jogador VENCEDOR serão guardadas.")
    print(f"Jogos a simular: {num_games} | Iterações: {iterations}")
    
    header = [f"c_{r}_{c}" for r in range(6) for c in range(7)]
    header.append("move")
    
    start_time = time.time()
    total_moves = 0
    jogos_uteis = 0
    
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        
        for i in range(num_games):
            board = Board()
            state_history = {board.get_state(): 1}
            game_over = False
            winner = None
            
            game_memory = [] 
            
            while not game_over:
                flat_state = get_flat_board(board)
                piece = board.current_player
                
                move = model_func(board, iterations=iterations)
                move_str = f"{move[0]}_{move[1]}"
                
                game_memory.append((piece, flat_state + [move_str]))
                
                opponent_piece = 'O' if piece == 'X' else 'X'
                
                if move[0] == "pop":
                    board.pop_piece(move[1])
                    cw, ow = board.check_win(piece), board.check_win(opponent_piece)
                    if cw and ow:
                        winner = piece
                        game_over = True
                    elif cw:
                        winner = piece
                        game_over = True
                    elif ow:
                        winner = opponent_piece
                        game_over = True
                else:
                    board.drop_piece(move[1], piece)
                    if board.check_win(piece):
                        winner = piece
                        game_over = True

                if not game_over:
                    new_state = board.get_state()
                    if state_history.get(new_state, 0) >= 3 or not board.get_valid_moves():
                        game_over = True
                    state_history[new_state] = state_history.get(new_state, 0) + 1
                    board.switch_player()
            
            if winner is not None:
                jogos_uteis += 1
                for player_who_moved, row_data in game_memory:
                    if player_who_moved == winner:
                        writer.writerow(row_data)
                        total_moves += 1
                    
            if (i + 1) % 5 == 0 or i == num_games - 1:
                print(f" -> Jogos processados: {i+1}/{num_games}... (Jogos com vencedor: {jogos_uteis})")

    elapsed = time.time() - start_time
    print(f"\n✅ Dataset '{filename}' gerado com sucesso!")
    print(f"📊 Total de jogadas VENCEDORAS gravadas: {total_moves}")
    print(f"⏱️ Tempo de processamento: {elapsed:.1f} segundos\n")

A função *run_data_set_generator* coleta os parâmetros selecionados e inicia a produção do dataset.

In [ ]:
def run_data_set_generator(num_games, iterations, filename):
    print(f"⚙️ A preparar configuração: Professor definido como MCTS Heurístico.")
    
    func = mcts_best_move
    nome = "MCTS Heurístico"
    
    generate_dataset_winners_only(
        model_func=func, 
        model_name=nome, 
        num_games=num_games, 
        iterations=iterations,
        filename=filename
    )


#run_data_set_generator(num_games=10, iterations=10000, filename="dataset_winners_test.csv")

A expectativa é que para um grande número de iterações e um dataset suficientemente grande, a qualidade e quantidade de seus dados poderão treinar uma árvore de decisão de modo que faça um bom movimento pelo menos em 50% das vezes, considerando que há cenários em que se jogado aleatoriamente, as chances de acerto de jogada ideal seria de aproximadamente 9%.

### ID3 vs MCTS Vanilla.

Com os datasets gerados e as árvores de decisão treinadas e exportas, é possível fazer o teste de comportamento da árvore de decisão em um cenário real de jogo. para isso, é necessário carregar o modelo de árvore desejado para essa batalha, a função *load_id3_model* faz a leitura do arquivo que guarda o dicionário aninhado.

In [ ]:
import json

def load_id3_model(filepath):
    with open(filepath, 'r') as f:
        return json.load(f)

A função *criar_agente_id3* funciona como uma "fábrica de jogadores". Como cada ficheiro JSON contém uma inteligência diferente (uma para cada profundidade), esta função isola o modelo escolhido dentro de um agente específico, permitindo que o sistema de simulação o trate como um jogador independente. Ela é essencial para que possamos instanciar e confrontar múltiplas versões da árvore de decisão simultaneamente ou em sequência, sem que os dados de um modelo interfiram no comportamento do outro durante a execução da arena.

In [ ]:
import random

def create_agent_id3(model):
    def agent(board):
        node = model
        
        while isinstance(node, dict):
            feature = list(node.keys())[0]
            parts = feature.split('_')
            r, c = int(parts[1]), int(parts[2])
            
            current_value = board.grid[r][c]
            if current_value == ' ':
                current_value = 'V'
            
            if current_value in node[feature]:
                node = node[feature][current_value]
            else:
                valid_moves = board.get_valid_moves()
                return random.choice(valid_moves)

        if isinstance(node, str) and "_" in node:
            acao, col = node.split("_")
            move = (acao, int(col))       
            return move

        return node
    return agent

A função *run_arena_tournament* é responsável por gerir e automatizar o ciclo completo de testes comparativos entre os diferentes modelos ID3 e o algoritmo MCTS Vanilla. Ela itera sobre uma lista de profundidades predefinidas, carrega dinamicamente os respetivos ficheiros de dados e utiliza a fábrica de agentes para configurar os confrontos na arena, alternando estrategicamente a ordem de jogo (quem começa a partida) para neutralizar qualquer viés de primeiro jogador. No final de cada bateria de simulações, a função compila as métricas de vitórias e empates diretamente num DataFrame do Pandas, gerando uma tabela comparativa estruturada que permite diagnosticar com precisão a evolução do desempenho da árvore de decisão à medida que o seu limite de profundidade aumenta.

In [ ]:
def run_arena_tournament(depths, matches_per_scenario=50):
    print(f"⚔️ STARTING MEGA TOURNAMENT ({matches_per_scenario} matches per scenario) ⚔️\n")
    results = []
    
    vanilla_name = "MCTS_VANILLA"
    
    for prof in depths:
        file_name = f"modelo_POPOUT_depth_{prof}.json"
        id3_name = f"ID3_Depth_{prof}"
        
        print(f"Processing matches for depth {prof}...")
        
        try:
            current_model = load_id3_model(file_name)
            id3_agent = create_agent_id3(current_model)
            
            vanilla_wins_s1, id3_wins_s1, draws_s1 = 0, 0, 0
            
            for _ in range(matches_per_scenario):
                res = simulate_match(vanilla_name, mcts_vanilla_best_move, id3_name, id3_agent)
                if res == vanilla_name: vanilla_wins_s1 += 1
                elif res == id3_name: id3_wins_s1 += 1
                else: draws_s1 += 1
                    
            results.append({
                "Depth": prof,
                "Player 1 (X)": vanilla_name,
                "Player 2 (O)": id3_name,
                "P1 Wins": vanilla_wins_s1,
                "P2 Wins": id3_wins_s1,
                "Draws": draws_s1
            })
            
            id3_wins_s2, vanilla_wins_s2, draws_s2 = 0, 0, 0
            
            for _ in range(matches_per_scenario):
                res = simulate_match(id3_name, id3_agent, vanilla_name, mcts_vanilla_best_move)
                if res == id3_name: id3_wins_s2 += 1
                elif res == vanilla_name: vanilla_wins_s2 += 1
                else: draws_s2 += 1
                    
            results.append({
                "Depth": prof,
                "Player 1 (X)": id3_name,
                "Player 2 (O)": vanilla_name,
                "P1 Wins": id3_wins_s2,
                "P2 Wins": vanilla_wins_s2,
                "Draws": draws_s2
            })
            
        except FileNotFoundError:
            print(f"  -> ERROR: File '{file_name}' not found. Skipping...\n")
            
    df_results = pd.DataFrame(results)
    
    print("\n✅ TOURNAMENT CONCLUDED!")
    display(HTML("<br><b>Final Results Table:</b>"))
    display(df_results)
    
    return df_results

depth_list = [2, 6, 10, 14, 18, 22, 26, 30, "Sem_Limite"]

#df_final = run_arena_tournament(depth_list, matches_per_scenario=50)

In [ ]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor

def _jogar_uma_partida(args):
    nome_c1, c1, nome_c2, c2, iterations, c1_e_X = args

    f_c1 = lambda b: mcts_vanilla_best_move(b, iterations=iterations, c=c1)
    f_c2 = lambda b: mcts_vanilla_best_move(b, iterations=iterations, c=c2)

    if c1_e_X:
        vencedor = simulate_match(nome_c1, f_c1, nome_c2, f_c2)
    else:
        vencedor = simulate_match(nome_c2, f_c2, nome_c1, f_c1)

    if vencedor == nome_c1:
        return 'c1'
    elif vencedor == nome_c2:
        return 'c2'
    else:
        return 'empate'


def avaliar_confronto_c(c1, c2, num_jogos_por_lado=50, iterations=2000, num_cores=4):
    nome_c1 = f"MCTS_C_{c1:.4f}"
    nome_c2 = f"MCTS_C_{c2:.4f}"

    tarefas = []
    for _ in range(num_jogos_por_lado):
        tarefas.append((nome_c1, c1, nome_c2, c2, iterations, True))
    for _ in range(num_jogos_por_lado):
        tarefas.append((nome_c1, c1, nome_c2, c2, iterations, False))

    print(f"  -> {len(tarefas)} jogos no total ({num_cores} threads)...")

    with ThreadPoolExecutor(max_workers=num_cores) as executor:
        resultados = list(executor.map(_jogar_uma_partida, tarefas))

    vitorias_c1 = sum(1 for r in resultados if r == 'c1')
    vitorias_c2 = sum(1 for r in resultados if r == 'c2')
    empates     = sum(1 for r in resultados if r == 'empate')

    print(f"  Resultados: {nome_c1}: {vitorias_c1} | {nome_c2}: {vitorias_c2} | Empates: {empates}")

    if vitorias_c2 != vitorias_c1:
        return vitorias_c2 > vitorias_c1
    else:
        return abs(c2 - 1.414) <= abs(c1 - 1.414)


def otimizar_c_busca_binaria(low=1.0, high=2.0, max_passos=5, iterations=2000, num_jogos_por_lado=50, num_cores=4):
    print(f"====== INICIANDO BUSCA DO VALOR IDEAL DE C NO INTERVALO [{low}, {high}] ======")
    print(f"       {num_jogos_por_lado*2} jogos/passo | {iterations} iterações | {num_cores} threads\n")

    for passo in range(1, max_passos + 1):
        meio  = (low + high) / 2
        delta = (high - low) * 0.1
        c1    = meio - delta
        c2    = meio + delta

        print(f"Passo {passo}/{max_passos}: Intervalo atual [{low:.4f}, {high:.4f}]")
        print(f"Testando C1 = {c1:.4f} vs C2 = {c2:.4f}")

        if avaliar_confronto_c(c1, c2, num_jogos_por_lado=num_jogos_por_lado,
                                iterations=iterations, num_cores=num_cores):
            print(f"👉 C2 venceu. Ajustando limite inferior para {meio:.4f}\n")
            low = meio
        else:
            print(f"👉 C1 venceu. Ajustando limite superior para {meio:.4f}\n")
            high = meio

    c_final = (low + high) / 2
    print(f"====== BUSCA CONCLUÍDA ======")
    print(f"O valor aproximado para o C ideal é: {c_final:.4f}")
    return c_final


melhor_c = otimizar_c_busca_binaria(
    low=1.0,
    high=2.0,
    max_passos=5,
    iterations=10000,
    num_jogos_por_lado=50,
    num_cores=10
)